In [1]:
import sys
from pathlib import Path
import pandas as pd

sys.path.append(str(Path("..").resolve() / "src"))

from yellow_taxi.extract_data import extract_yellow_tripdata

path_yellow_trip = Path("..") / "data" / "yellow_tripdata_2016-03.csv"

df = extract_yellow_tripdata(path_yellow_trip)
df['tpep_dropoff_datetime'] = pd.to_datetime(
                                df['tpep_dropoff_datetime'],
                                format='%Y-%m-%d %H:%M:%S',
                                errors='coerce'
                            )
df['tpep_pickup_datetime'] = pd.to_datetime(
                                df['tpep_pickup_datetime'],
                                format='%Y-%m-%d %H:%M:%S',
                                errors='coerce'
                            )

df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,pickup_longitude,pickup_latitude,RatecodeID,store_and_fwd_flag,dropoff_longitude,dropoff_latitude,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount
0,1,2016-03-01,2016-03-01 00:07:55,1,2.50,-73.976746,40.765152,1,N,-74.004265,40.746128,1,9.0,0.5,0.5,2.05,0.00,0.3,12.35
1,1,2016-03-01,2016-03-01 00:11:06,1,2.90,-73.983482,40.767925,1,N,-74.005943,40.733166,1,11.0,0.5,0.5,3.05,0.00,0.3,15.35
2,2,2016-03-01,2016-03-01 00:31:06,2,19.98,-73.782021,40.644810,1,N,-73.974541,40.675770,1,54.5,0.5,0.5,8.00,0.00,0.3,63.80
3,2,2016-03-01,2016-03-01 00:00:00,3,10.78,-73.863419,40.769814,1,N,-73.969650,40.757767,1,31.5,0.0,0.5,3.78,5.54,0.3,41.62
4,2,2016-03-01,2016-03-01 00:00:00,5,30.43,-73.971741,40.792183,3,N,-74.177170,40.695053,1,98.0,0.0,0.0,0.00,15.50,0.3,113.80


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12210952 entries, 0 to 12210951
Data columns (total 19 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   VendorID               int64         
 1   tpep_pickup_datetime   datetime64[us]
 2   tpep_dropoff_datetime  datetime64[us]
 3   passenger_count        int64         
 4   trip_distance          float64       
 5   pickup_longitude       float64       
 6   pickup_latitude        float64       
 7   RatecodeID             int64         
 8   store_and_fwd_flag     str           
 9   dropoff_longitude      float64       
 10  dropoff_latitude       float64       
 11  payment_type           int64         
 12  fare_amount            float64       
 13  extra                  float64       
 14  mta_tax                float64       
 15  tip_amount             float64       
 16  tolls_amount           float64       
 17  improvement_surcharge  float64       
 18  total_amount           float64 

In [3]:
# Converter colunas de data
df['pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])
df['dropoff_datetime'] = pd.to_datetime(df['tpep_dropoff_datetime'])

# Verificar valores nulos
print("=== VALORES NULOS ===")
null_counts = df.isnull().sum()
print(null_counts[null_counts > 0] if null_counts.sum() > 0 else "Nenhum valor nulo encontrado")

=== VALORES NULOS ===
Nenhum valor nulo encontrado


In [4]:
# Detectar anomalias
total = len(df)

# Distância inválida
invalid_distance = (df['trip_distance'] <= 0) | (df['trip_distance'] > 100)

# Número de passageiros inválido
invalid_passengers = (df['passenger_count'] <= 0) | (df['passenger_count'] > 6)

# Tarifa negativa
invalid_fare = df['fare_amount'] < 0

# Duração inválida (pickup depois do dropoff)
invalid_duration = df['pickup_datetime'] > df['dropoff_datetime']

# Coordenadas fora de NYC (latitude ~40.4-41.0, longitude ~-74.3 a -73.7)
invalid_pickup_coords = (
    (df['pickup_latitude'] < 40.4) | (df['pickup_latitude'] > 41.0) |
    (df['pickup_longitude'] < -74.3) | (df['pickup_longitude'] > -73.7)
)
invalid_dropoff_coords = (
    (df['dropoff_latitude'] < 40.4) | (df['dropoff_latitude'] > 41.0) |
    (df['dropoff_longitude'] < -74.3) | (df['dropoff_longitude'] > -73.7)
)

print("=== ANOMALIAS DETECTADAS ===")
print(f"Distância inválida (<=0 ou >100mi): {invalid_distance.sum():,} ({invalid_distance.sum()/total*100:.2f}%)")
print(f"Passageiros inválidos (<=0 ou >6):  {invalid_passengers.sum():,} ({invalid_passengers.sum()/total*100:.7f}%)")
print(f"Tarifa negativa:                    {invalid_fare.sum():,} ({invalid_fare.sum()/total*100:.4f}%)")
print(f"Duração inválida:                   {invalid_duration.sum():,} ({invalid_duration.sum()/total*100:.7f}%)")
print(f"Coordenadas pickup fora de NYC:     {invalid_pickup_coords.sum():,} ({invalid_pickup_coords.sum()/total*100:.2f}%)")
print(f"Coordenadas dropoff fora de NYC:    {invalid_dropoff_coords.sum():,} ({invalid_dropoff_coords.sum()/total*100:.2f}%)")

=== ANOMALIAS DETECTADAS ===
Distância inválida (<=0 ou >100mi): 71,225 (0.58%)
Passageiros inválidos (<=0 ou >6):  678 (0.0055524%)
Tarifa negativa:                    4,581 (0.0375%)
Duração inválida:                   109 (0.0008926%)
Coordenadas pickup fora de NYC:     183,816 (1.51%)
Coordenadas dropoff fora de NYC:    178,754 (1.46%)


In [7]:
# Remover registros inválidos
invalid_mask = (
    invalid_distance |
    invalid_passengers |
    invalid_fare |
    invalid_duration |
    invalid_pickup_coords |
    invalid_dropoff_coords
)

df_cleaned = df[~invalid_mask].copy()
itens_removed = total - len(df_cleaned)

print(f'Registros antes da limpeza: {total:,}')

print(f'Itens removidos: {itens_removed:,} ({itens_removed/total*100:.2f}%)')

print(f'Novo total: {len(df_cleaned)}')

Registros antes da limpeza: 12,210,952
Itens removidos: 255,359 (2.09%)
Novo total: 11955593
